# OpenAPI-compatible client test

**QE Perspective:** We validate the OpenAPI-compatible Responses API when called via the OpenAI client (e.g. `base_url` pointing at OGX). We ensure **positive** (valid request returns completed status and correct content), **negative** (invalid model is rejected with 4xx or non-completed status), and **edge** (empty input and missing config are handled predictably). This keeps the OpenAPI contract and error behavior testable.

- **Positive:** Valid model and input → status completed, output contains expected answer (e.g. "4").
- **Negative:** Non-existent model → API error or non-completed response.
- **Edge:** Empty input → either error or valid response with output; missing config → assert fail fast.

Config: `BASE_URL`, `MODEL`. Run via pytest.


## Setup

Load config (base_url, model) from env and the shared `response_text` helper from `scripts.helpers`; create the OpenAI client pointed at OGX. Assert base_url and model are set (edge: missing config).


In [ ]:
import os
from scripts.helpers import response_text

base_url = os.environ.get("BASE_URL", "http://localhost:8321")
# OpenAI client needs /v1 suffix
openai_base_url = base_url.rstrip("/") + "/v1"
model = os.environ.get("INFERENCE_MODEL")

# Edge: fail fast if config is missing
assert base_url, (
    "base_url is empty; set BASE_URL or run via run-tests-with-providers.sh"
)
assert model, "model is empty; set MODEL or run via run-tests-with-providers.sh"

In [3]:
from openai import OpenAI

openai_client = OpenAI(api_key="no-key-needed", base_url=openai_base_url)

In [4]:
# --- Positive: valid request ---
response = openai_client.responses.create(
    model=model,
    input="What is 2 + 2? Reply with the number only.",
)

assert response.status == "completed", (
    f"Expected status completed, got {response.status}"
)
assert response.output, "Expected at least one output"
text = response_text(response)
assert "4" in text, f"Expected 4 in response, got: {text}"

## Negative: invalid model

Call with a non-existent model; expect the API to return an error (e.g. 404 or 400).


In [5]:
from openai import APIError

invalid_model_handled = False
try:
    r = openai_client.responses.create(
        model="model",
        input="Hello",
    )
    # Some backends return 200 with status failed / error set
    invalid_model_handled = r.status != "completed" or getattr(r, "error", None)
except APIError as e:
    invalid_model_handled = e.status_code in (400, 404, 422)
assert invalid_model_handled, (
    "Invalid model must be rejected (4xx or non-completed response) "
)

## Edge: empty input

Empty or minimal input; server may return error or empty output. We accept either: error raised or response with empty/failed status.


In [6]:
from openai import APIError

empty_ok = False
try:
    r = openai_client.responses.create(model=model, input="")
    empty_ok = r.status in ("completed", "failed", "cancelled") and r.output is not None
except APIError:
    empty_ok = True  # Rejecting empty input is also acceptable
assert empty_ok, "Empty input must yield either a valid response or APIError"